In [2]:
from pymongo import MongoClient
import pandas as pd

client = MongoClient()
db = client.SAE

In [3]:
# Q1 : Lister les clients n’ayant jamais effecuté une commande
Q1 = pd.DataFrame(list(
    db.customers.aggregate([
        { "$match": { "orders": { "$eq": [] } } }
    ])
))
Q1

,_id,customerNumber,customerName,contactLastName,contactFirstName,phone,addressLine1,addressLine2,city,state,postalCode,country,salesRepEmployeeNumber,creditLimit,orders
0,693c0cddd3467b0721d85e12,125,Havel & Zbyszek Co,Piestrzeniewicz,Zbyszek,(26) 642-7555,ul. Filtrowa 68,NULL,Warszawa,NULL,01-012,Poland,NULL,0.0,[]
1,693c0cddd3467b0721d85e20,168,American Souvenirs Inc,Franco,Sue,2035557845,149 Spinnaker Dr.,Suite 101,New Haven,CT,97823,USA,1286,0.0,[]
2,693c0cddd3467b0721d85e21,169,Porto Imports Co.,de Castro,Isabel,(1) 356-5555,Estrada da saúde n. 58,NULL,Lisboa,NULL,1756,Portugal,NULL,0.0,[]
3,693c0cddd3467b0721d85e30,206,"Asian Shopping Network, Co",Walker,Brydey,+612 9411 1555,"Penthouse Level, Suntec Tower Three, 8 Temasek",NULL,Singapore,NULL,038988,Singapore,NULL,0.0,[]
4,693c0cddd3467b0721d85e35,223,Natürlich Autos,Kloss,Horst,0372-555188,Taucherstraße 10,NULL,Cunewalde,NULL,01307,Germany,NULL,0.0,[]
5,693c0cddd3467b0721d85e38,237,ANG Resellers,Camino,Alejandra,(91) 745 6555,"Gran Vía, 1",NULL,Madrid,NULL,28001,Spain,NULL,0.0,[]
6,693c0cddd3467b0721d85e3c,247,Messner Shopping Network,Messner,Renate,069-0555984,Magazinweg 7,NULL,Frankfurt,NULL,60528,Germany,NULL,0.0,[]
7,693c0cddd3467b0721d85e42,273,"Franken Gifts, Co",Franken,Peter,089-0877555,Berliner Platz 43,NULL,München,NULL,80805,Germany,NULL,0.0,[]
8,693c0cddd3467b0721d85e47,293,BG&E Collectables,Pon,Ed,+41 26 425 50 01,Rte des Arsenaux 41,NULL,Fribourg,NULL,1700,Switzerland,NULL,0.0,[]
9,693c0cddd3467b0721d85e4a,303,Schuyler Imports,Schuyler,Bradley,+31 20 491 9555,Kingsfordweg 151,NULL,Amsterdam,NULL,1043 GR,Netherlands,NULL,0.0,[]


In [4]:
# Q2 : Pour chaque employé, le nombre de clients, le nombre de commandes et le montant total de celles-ci
Q2 = pd.DataFrame(list(
    db.offices.aggregate([
        { "$unwind": "$employees" },
        {
            "$lookup": {
                "from": "customers",
                "localField": "employees.employeeNumber",
                "foreignField": "salesRepEmployeeNumber",
                "as": "clients"
            }
        },
        {
            "$project": {
                "_id": 0,
                "employeeNumber": "$employees.employeeNumber",
                "firstName": "$employees.firstName",
                "lastName": "$employees.lastName",
                "nbClients": { "$size": "$clients" },
                "nbOrders": {
                    "$sum": {
                        "$map": {
                            "input": "$clients",
                            "as": "c",
                            "in": { "$size": "$$c.orders" }
                        }
                    }
                },
                "totalAmount": {
                    "$sum": {
                        "$map": {
                            "input": "$clients",
                            "as": "c",
                            "in": { "$sum": "$$c.orders.total" }
                        }
                    }
                }
            }
        }
    ])
))
Q2

,employeeNumber,firstName,lastName,nbClients,nbOrders,totalAmount
0,1002,Diane,Murphy,0,0,0
1,1056,Mary,Patterson,0,0,0
2,1076,Jeff,Firrelli,0,0,0
3,1143,Anthony,Bow,0,0,0
4,1165,Leslie,Jennings,6,34,0
5,1166,Leslie,Thompson,6,14,0
6,1188,Julie,Firrelli,6,14,0
7,1216,Steve,Patterson,6,18,0
8,1286,Foon Yue,Tseng,7,17,0
9,1323,George,Vanauf,8,22,0


In [ ]:
# Q3 : Idem pour chaque bureau (nombre de clients, nombre de commandes et montant total), avec en plus le nombre de clients d’un pays différent, s’il y en a
Q3 = pd.DataFrame(list(
    db.offices.aggregate([
        {
            "$lookup": {
                "from": "customers",
                "localField": "officeCode",
                "foreignField": "salesRepEmployeeNumber",
                "as": "clients"
            }
        },
        {
            "$project": {
                "_id": 0,
                "officeCode": 1,
                "city": 1,
                "country": 1,
                "nbClients": { "$size": "$clients" },
                "nbOrders": {
                    "$sum": {
                        "$map": {
                            "input": "$clients",
                            "as": "c",
                            "in": { "$size": "$$c.orders" }
                        }
                    }
                },
                "totalAmount": {
                    "$sum": {
                        "$map": {
                            "input": "$clients",
                            "as": "c",
                            "in": { "$sum": "$$c.orders.total" }
                        }
                    }
                },
                "nbForeignClients": {
                    "$size": {
                        "$filter": {
                            "input": "$clients",
                            "as": "c",
                            "cond": { "$ne": [ "$$c.country", "$country" ] }
                        }
                    }
                }
            }
        }
    ])
))
Q3


,officeCode,city,country,nbClients,nbOrders,totalAmount,nbForeignClients
0,1.0,San Francisco,USA,0,0,0,0
1,2.0,Boston,USA,0,0,0,0
2,3.0,NYC,USA,0,0,0,0
3,4.0,Paris,France,0,0,0,0
4,5.0,Tokyo,Japan,0,0,0,0
5,6.0,Sydney,Australia,0,0,0,0
6,7.0,London,UK,0,0,0,0


In [7]:
#Q4 : Pour chaque produit, donner le nombre de commandes, la quantité totale commandée, et le nombre de clients différents
Q4 = pd.DataFrame(list(
    db.products.aggregate([
        {
            "$lookup": {
                "from": "orders",
                "let": { "prodCode": "$productCode" },
                "pipeline": [
                    { "$unwind": "$details" },
                    {
                        "$match": {
                            "$expr": { "$eq": [ "$details.productCode", "$$prodCode" ] }
                        }
                    }
                ],
                "as": "orders"
            }
        },

        { "$unwind": "$orders" },

        {
            "$group": {
                "_id": "$productCode",
                "productName": { "$first": "$productName" },
                "nb_commandes": { "$addToSet": "$orders.orderNumber" },
                "quantite_totale": { "$sum": "$orders.details.quantityOrdered" },
                "nb_clients": { "$addToSet": "$orders.customerNumber" }
            }
        },

        {
            "$project": {
                "_id": 0,
                "productCode": "$_id",
                "productName": 1,
                "nb_commandes": { "$size": "$nb_commandes" },
                "quantite_totale": 1,
                "nb_clients": { "$size": "$nb_clients" }
            }
        }
    ])
))
Q4


""


In [32]:
#Q5 : Donner le nombre de commande pour chaque pays du client, ainsi que le montant total des commandes et le montant total payé : On veut conserver les clients n’ayant jamais commandé dans le résultat final
Q5 = pd.DataFrame(list(
    db.customers.aggregate([
        {
            "$lookup": {
                "from": "orders",
                "localField": "customerNumber",
                "foreignField": "customerNumber",
                "as": "orders"
            }
        },
        { "$unwind": { "path": "$orders", "preserveNullAndEmptyArrays": True } },

        { "$unwind": { "path": "$orders.details", "preserveNullAndEmptyArrays": True } },

        {
            "$lookup": {
                "from": "payments",
                "localField": "customerNumber",
                "foreignField": "customerNumber",
                "as": "payments"
            }
        },

        {
            "$group": {
                "_id": "$country",
                "nb_commandes": { "$addToSet": "$orders.orderNumber" },
                "montant_commandes": {
                    "$sum": {
                        "$ifNull": [
                            { "$multiply": ["$orders.details.quantityOrdered", "$orders.details.priceEach"] },
                            0
                        ]
                    }
                },
                "montant_paye": { "$sum": "$payments.amount" }
            }
        },

        {
            "$project": {
                "_id": 0,
                "country": "$_id",
                "nb_commandes": { "$size": { "$setDifference": ["$nb_commandes", [None]] }},
                "montant_commandes": 1,
                "montant_paye": 1
            }
        }
    ])
))
Q5


,montant_commandes,montant_paye,country,nb_commandes
0,0.00,0,Portugal,0
1,307463.70,0,Norway,9
2,224078.56,0,Canada,7
3,0.00,0,Russia,0
4,3627982.85,0,USA,112
5,0.00,0,Poland,0
6,220472.09,0,Germany,7
7,1215686.92,0,Spain,36
8,94015.73,0,Philippines,3
9,48784.36,0,Hong Kong,2


In [ ]:
#Q6 : On veut la table de contigence du nombre de commande entre la ligne de produits et le pays du client
Q6 = pd.DataFrame(list(
    db.orders.aggregate([
        { "$unwind": "$details" },
        {
            "$lookup": {
                "from": "customers",
                "localField": "customerNumber",
                "foreignField": "customerNumber",
                "as": "cust"
            }
        },
        { "$unwind": "$cust" },
        {
            "$group": {
                "_id": {
                    "productLine": "$details.productLine",
                    "country": "$cust.country"
                },
                "nbOrders": { "$sum": 1 }
            }
        }
    ])
))
Q6

,_id,nbOrders
0,"{'productLine': 'Planes', 'country': 'Germany'}",8
1,"{'productLine': 'Vintage Cars', 'country': 'Ge...",9
2,"{'productLine': 'Classic Cars', 'country': 'De...",34
3,"{'productLine': 'Motorcycles', 'country': 'New...",26
4,"{'productLine': 'Classic Cars', 'country': 'Fi...",38
...,...,...
121,"{'productLine': 'Planes', 'country': 'Norway'}",11
122,"{'productLine': 'Trains', 'country': 'UK'}",4
123,"{'productLine': 'Trucks and Buses', 'country':...",6
124,"{'productLine': 'Planes', 'country': 'Italy'}",35


In [9]:
#Q7 : On veut la même table croisant la ligne de produits et le pays du client, mais avec le montant total payé dans chaque cellule 
Q7 = pd.DataFrame(list(
    db.orders.aggregate([
        { "$unwind": "$details" },
        {
            "$lookup": {
                "from": "customers",
                "localField": "customerNumber",
                "foreignField": "customerNumber",
                "as": "cust"
            }
        },
        { "$unwind": "$cust" },
        {
            "$lookup": {
                "from": "payments",
                "localField": "customerNumber",
                "foreignField": "customerNumber",
                "as": "pays"
            }
        },
        {
            "$group": {
                "_id": {
                    "productLine": "$details.productLine",
                    "country": "$cust.country"
                },
                "totalPaid": { "$sum": { "$sum": "$pays.amount" } }
            }
        }
    ])
))
Q7


,_id,totalPaid
0,"{'productLine': 'Ships', 'country': 'Germany'}",69987.84
1,"{'productLine': 'Vintage Cars', 'country': 'USA'}",47090713.86
2,"{'productLine': 'Motorcycles', 'country': 'Hon...",48784.36
3,"{'productLine': 'Planes', 'country': 'Australia'}",3619902.87
4,"{'productLine': 'Trucks and Buses', 'country':...",1049392.71
...,...,...
121,"{'productLine': 'Planes', 'country': 'Belgium'}",33440.10
122,"{'productLine': 'Ships', 'country': 'Sweden'}",1225584.40
123,"{'productLine': 'Trucks and Buses', 'country':...",224160.56
124,"{'productLine': 'Classic Cars', 'country': 'Ir...",346538.58


In [29]:
# Q8 : Donner les 10 produits pour lesquels la marge moyenne est la plus importante (cf buyPrice et priceEach)
Q8 = pd.DataFrame(list(
    db.orders.aggregate([
        { "$unwind": "$details" },
        {
            "$group": {
                "_id": "$details.productCode",
                "productName": { "$first": "$details.productName" },
                "avgMargin": {
                    "$avg": {
                        "$subtract": [
                            "$details.priceEach",
                            "$details.buyPrice"
                        ]
                    }
                }
            }
        },
        { "$sort": { "avgMargin": -1 } },
        { "$limit": 10 }
    ])
))
Q8

,_id,productName,avgMargin
0,S10_1949,1952 Alpine Renault 1300,99.006429
1,S10_4698,2003 Harley-Davidson Eagle Drag Bike,95.235000
2,S18_3232,1992 Ferrari 360 Spider red,83.334906
3,S12_2823,2002 Suzuki XREO,83.201429
4,S18_2795,1928 Mercedes-Benz SSK,82.696786
5,S12_1108,2001 Ferrari Enzo,81.043704
6,S12_3891,1969 Ford Falcon,77.335926
7,S18_3685,1948 Porsche Type 356 Roadster,72.636800
8,S18_2870,1999 Indy 500 Monte Carlo SS,71.794400
9,S18_1749,1917 Grand Touring Sedan,70.432800


In [ ]:
#Q9 : Lister les produits (avec le nom et le code du client) qui ont été vendus à perte
Q9 = pd.DataFrame(list(
    db.orders.aggregate([
        { "$unwind": "$details" },
        {
            "$match": {
                "$expr": { "$lt": [ "$details.priceEach", "$details.buyPrice" ] }
            }
        },
        {
            "$lookup": {
                "from": "customers",
                "localField": "customerNumber",
                "foreignField": "customerNumber",
                "as": "customer"
            }
        },
        { "$unwind": "$customer" },
        {
            "$project": {
                "_id": 0,
                "productCode": "$details.productCode",
                "productName": "$details.productName",
                "customerNumber": "$customer.customerNumber",
                "customerName": "$customer.customerName",
                "priceEach": "$details.priceEach",
                "buyPrice": "$details.buyPrice"
            }
        }
    ])
))
Q9

,productCode,productName,customerNumber,customerName,priceEach,buyPrice
0,S10_4962,1962 LanciaA Delta 16V,363,Online Diecast Creations Co.,61.99,103.42
1,S18_2957,1934 Ford V8 Coupe,363,Online Diecast Creations Co.,29.87,34.35
2,S18_3136,18th Century Vintage Horse Carriage,363,Online Diecast Creations Co.,47.04,60.74
3,S12_3148,1969 Corvair Monza,181,Vitachrome Inc.,54.33,89.14
4,S18_2319,1964 Mercedec Tour Bus,181,Vitachrome Inc.,37.48,74.86
...,...,...,...,...,...,...
74,S10_4962,1962 LanciaA Delta 16V,276,"Anna's Decorations, Ltd",46.90,103.42
75,S12_1666,1958 Setra Bus,276,"Anna's Decorations, Ltd",63.20,77.90
76,S18_2949,1913 Ford Model T Speedster,276,"Anna's Decorations, Ltd",45.25,60.78
77,S18_2238,1998 Chrysler Plymouth Prowler,323,"Down Under Souveniers, Inc",69.81,101.51


In [ ]:
#Q10 : Lister les clients pour lesquels le montant total payé est inférieur aux montants totals des achats
Q10 = pd.DataFrame(list(
    db.customers.aggregate([
        {
            "$lookup": {
                "from": "payments",
                "localField": "customerNumber",
                "foreignField": "customerNumber",
                "as": "payments"
            }
        },
        {
            "$addFields": {
                "totalPaid": { "$sum": "$payments.amount" }
            }
        },

        { "$unwind": { "path": "$orders", "preserveNullAndEmptyArrays": True } },
        { "$unwind": { "path": "$orders.details", "preserveNullAndEmptyArrays": True } },
        
        {
            "$group": {
                "_id": {
                    "customerNumber": "$customerNumber",
                    "customerName": "$customerName"
                },
                "montant_achats": {
                    "$sum": {
                        "$ifNull": [
                            { "$multiply": [ "$orders.details.quantityOrdered", "$orders.details.priceEach" ] },
                            0
                        ]
                    }
                },
                "montant_paye": { "$first": "$totalPaid" }
            }
        },
        {
            "$project": {
                "_id": 0,
                "customerNumber": "$_id.customerNumber",
                "customerName": "$_id.customerName",
                "montant_achats": 1,
                "montant_paye": 1
            }
        },
    ])
))
Q10

,montant_achats,montant_paye,customerNumber,customerName
0,0,26479.26,198,Auto-Moto Classics Inc.
1,0,82751.08,112,Signal Gift Stores
2,0,49642.05,344,CAF Imports
3,0,111640.28,167,Herkku Gifts
4,0,0.00,348,"Asian Treasures, Inc."
...,...,...,...,...
117,0,0.00,237,ANG Resellers
118,0,72321.12,412,"Extreme Desk Decorations, Ltd"
119,0,57197.96,204,Online Mini Collectables
120,0,142874.25,146,"Saveley & Henriot, Co."
